In [ ]:
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import math

In [ ]:
class CausalSelfAttention(nn.Module):
  def __init__(self,config):
    super().__init__()
    assert config.n_embd % config.n_head == 0
    # key, query, value projection for all heads but in a batch
    self.c_attn = nn.Linear(config.n_embd, 3* config.n_embd)
    # output projection
    self.c_proj = nn.Linear(config.n_embd,config.n_embd)
    self.c_proj.NANOGPT_SCALE_INIT = 1
    self.n_head = config.n_head
    self.n_embd = config.n_embd
    bs = config.block_size
    self.register_buffer("bias", torch.tril(torch.ones(bs, bs)).view(1,1,bs,bs))


  def forward(self,x):
    B,T,C = x.size()
    qkv = self.c_attn(x)
    q,k,v = qkv.split(self.n_embd,dim=2)
    k = k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
    q = q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
    v = v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
    # att = (q @ k.transpose(-2,-1)) * (1.0/math.sqrt(k.size(-1)))
    # att = att.masked_fill(self.bias[:,:,:T,:T]==0,float('-inf'))
    # att = F.softmax(att,dim=-1)
    # y = att @ v
    y = F.scaled_dot_product_attention(q,k,v,is_casual=True)
    y = y.transpose(1,2).contiguous().view(B,T,C)
    y = self.c_proj(y)
    return y

class MLP(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.config = config
    self.c_fc = nn.Linear(config.n_embd,4*config.n_embd)
    self.gelu = nn.GELU(approximate='tanh')
    self.c_proj = nn.Linear(4 * config.n_embd,config.n_embd)
    self.c_proj.NANOGPT_SCALE_INIT = 1

  def forward(self,x):
    x = self.c_fc(x)
    x = self.gelu(x)
    x = self.c_proj(x)
    return x

class Block(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.config = config
    self.ln_1 = nn.LayerNorm(config.n_embd)
    self.attn = CausalSelfAttention(config)
    self.ln_2 = nn.LayerNorm(config.n_embd)
    self.mlp = MLP(config)

  def forward(self,x):
    x = x + self.attn(self.ln_1(x))
    x = x + self.mlp(self.ln_2(x)) # this is the feed forward network (ffn)
    return x

@dataclass
class GPTConfig:
    block_size: int = 1024 # max sequence length
    vocab_size: int = 50257 # number of tokens: 50,000 BPE merges + 256 bytes tokens + 1 <|endoftext|> token
    n_layer: int = 12 # number of layers
    n_head: int = 12 # number of heads
    n_embd: int = 768 # embedding dimension

class GPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.config = config

    self.transformer = nn.ModuleDict(dict(
        wte = nn.Embedding(config.vocab_size, config.n_embd),
        wpe = nn.Embedding(config.block_size, config.n_embd),
        h = nn.ModuleList(Block(config) for _ in range(config.n_layer)),
        ln_f = nn.LayerNorm(config.n_embd)
    ))
    self.lm_head=nn.Linear(config.n_embd,config.vocab_size,bias = False)

    # weight sharing scheme
    self.transformer.wte.weight = self.lm_head.weight

    # init params
    self.apply(self._init_weights)

  def _init_weights(self,module):
    if isinstance(module,nn.Linear):
      std = 0.02
      if hasattr(module,'NANOGPT_SCALE_INIT'):
        std *= (2 * self.config_n_layer)** -0.5
      torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module,nn.Embedding):
      torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)


  def forward(self,idx,targets=None):
    # idx of shape (B,T) (batch and time dimension)
    B, T = idx.size()
    assert T<= self.config.block_size, "Cannot forward, model block size is exhausted"
    pos = torch.arange(0,T,dtype=torch.long,device=idx.device)
    pos_emb = self.transformer.wpe(pos) ## positional embedding
    tok_emb = self.transformer.wte(idx) ## token embedding
    x = tok_emb + pos_emb
    for block in self.transformer.h:
      x = block(x)
    x = self.transformer.ln_f(x)
    logits = self.lm_head(x)
    loss = None
    if targets is not None:
      loss = F.cross_entropy(logits.view(-1,logits.size(-1)),targets.view(-1))
    return logits,loss

  @classmethod
  def from_pretrained(cls, model_type):
    """Loads pretrained GPT-2 model weights from huggingface"""
    assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
    from transformers import GPT2LMHeadModel
    print("loading weights from pretrained gpt: %s" % model_type)

    # n_layer, n_head and n_embd are determined from model_type
    config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
    }[model_type]
    config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
    config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
    # create a from-scratch initialized minGPT model
    config = GPTConfig(**config_args)
    model = GPT(config)
    sd = model.state_dict()
    sd_keys = sd.keys()
    sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

    # init a huggingface/transformers model
    model_hf = GPT2LMHeadModel.from_pretrained(model_type)
    sd_hf = model_hf.state_dict()

    # copy while ensuring all of the parameters are aligned and match in names and shapes
    sd_keys_hf = sd_hf.keys()
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
    transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
    # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
    # this means that we have to transpose these weights when we import them
    assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
    for k in sd_keys_hf:
      if any(k.endswith(w) for w in transposed):
        # special treatment for the Conv1D weights we need to transpose
        assert sd_hf[k].shape[::-1] == sd[k].shape
        with torch.no_grad():
          sd[k].copy_(sd_hf[k].t())
      else:
        # vanilla copy over the other parameters
        assert sd_hf[k].shape == sd[k].shape
        with torch.no_grad():
          sd[k].copy_(sd_hf[k])

    return model

  def configure_optimizers(self,weight_decay,learning_rate,device):
    param_dict = {pn:p for pn,p in self.named_parameters()}
    param_dict = {pn:p for pn,p in param_dict.items() if p.requires_grad}
    decay_params = [p for n,p in param_dict.items() if p.dim()>=2]
    nodecay_params = [p for n,p in param_dict.items() if p.dim() < 2]
    optim_groups = [
        {'params':decay_params,'weight_decay':weight_decay},
        {'params':nodecay_params,'weight_decay':0.0}
    ]
    num_decay_params = sum(p.numel() for p in decay_params)
    num_nodecay_params = sum(p.numel() for p in nodecay_params)
    print(f"num decayed params: {num_decay_params}")
    print(f"num non-decayed params: {num_nodecay_params}")

    fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
    use_fused = fused_available and device.type == 'cuda'
    print(f"using fused adamw: {use_fused}")
    optimizer = torch.optim.AdamW(optim_groups,lr=learning_rate,betas=(0.9,0.95),eps=1e-8,fused=use_fused)
    return optimizer

In [ ]:
import tiktoken

class DataLoaderLite:
  def __init__(self,B,T):
    self.B = B
    self.T = T

    with open("input.txt","r") as f:
      text = f.read()

    enc = tiktoken.get_encoding('gpt2')
    tokens = enc.encode(text)
    self.tokens = torch.tensor(tokens)
    print(f"loaded {len(self.tokens)} tokens")
    print(f"1 epoch  = {len(self.tokens)//(self.B*self.T)} steps")

    # state
    self.current_position = 0

  def next_batch(self):
    B , T = self.B, self.T
    buf = self.tokens[self.current_position:self.current_position+B*T+1]
    x = buf[:-1].view(B,T)
    y = buf[1:].view(B,T)
    self.current_position += B*T
    if self.current_position >= len(self.tokens):
      self.current_position = 0
    return x,y

In [ ]:
batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 284
n_head = 6
n_layer = 6
dropout = 0.2

In [ ]:
total_batch_size = 524288 # 2**19 aprox 0.5M in number of tokens
B = 16 # micro batch size
T = 1024 ## sequence length

assert total_batch_size % (B*T) == 0, "make sure total_batch_size is divisible by B*T"
grad_accum_steps = total_batch_size // (B*T)
print(f"Total desired batch size: {total_batch_size}")
print(f"=> calculated gradient accumulation steps: {grad_accum_steps}")

In [ ]:
#model = GPT.from_pretrained("gpt2")
model = GPT(GPTConfig(vocab_size=50304))

In [ ]:
num_return_sequences = 5
max_length = 30
model = model.to(device)
model = torch.compile(model)
model.eval()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode("Hello, I'm a language model,")
## unsqueeze to add a batch dimension
tokens = torch.tensor(tokens,dtype=torch.long,device="cuda").unsqueeze(dim=0).repeat(num_return_sequences,1).to(device)
x = tokens.to('cuda')

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
while x.size(1) < max_length:
  # forward the model to get the logits
  with torch.no_grad():
    logits,_ = model(x)
    logits = logits[:,-1,:]
    probs = F.softmax(logits,dim=-1)
    topk_probs,topk_indices = torch.topk(probs,50,dim=-1)
    ix = torch.multinomial(topk_probs,1)
    xcol = torch.gather(topk_indices,dim=-1,index=ix)
    x = torch.cat((x,xcol),dim=1)

In [ ]:
for i in range(num_return_sequences):
  tokens = x[i,:max_length].tolist()
  decoded= enc.decode(tokens)
  print(decoded)

Hello, I'm a language model,Styfeld answ remedy announcesspective McCoyhp Movement Pets bandwidth Flipurnedо� Sly Pushpole 308iddenperor efforts Daly
Hello, I'm a language model, suscept Lect broaden267 midrangesubestation sparkling 186Mehello {{controlleruncturetight Stadium accurate Goodwin refining sits indefinitetml
Hello, I'm a language model,THE breeding enjoying693 evidence bins MillionsDun Mull applauseRot addingaba Achilles MLA Leadership whirlwind Hod APRinsky screen Arrows
Hello, I'm a language model, suspensionんwhe skatingnothing lingeringiper reluctant squadroncal Souls elaborate 336 stocks Cruisericoneashiiku clitor attracting020api
Hello, I'm a language model, ble shale cirバ hypothesesDX campength manip dmg 2004 Drawing assailants clinical delivers Right PercentageBuildingundleoglucyclopediaMAR


In [ ]:
## start training process
!wget https://raw.githubusercontent.com/karpathy/ng-video-lecture/refs/heads/master/input.txt

--2026-01-25 17:25:46--  https://raw.githubusercontent.com/karpathy/ng-video-lecture/refs/heads/master/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-01-25 17:25:47 (37.8 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
with open('input.txt','r',encoding='utf-8') as f:
  text = f.read()

In [ ]:
import torch
B,T = 4 ,32
train_loader = DataLoaderLite(B,T)
model = GPT(GPTConfig())
model.to(device)

loaded 338025 tokens
1 epoch  = 2640 steps


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
import math
max_lr = 6e-4
min_lr = max_lr * 0.1
warmup_steps = 10
max_steps = 50
def get_lr(it):
  if it < warmup_steps:
    return max_lr * (it+1) / warmup_steps

  if it > max_steps:
    return min_lr
  decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
  assert 0 <= decay_ratio <= 1
  coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
  return min_lr + coeff * (max_lr - min_lr)

In [ ]:
#optimizer = torch.optim.AdamW(model.parameters(),lr=learning,betas=(0.9,0.95),eps=1e-8)
optimizer = model.configure_optimizers(weight_decay=0.1,learning_rate=6e-4,device=device)
for step in range(max_steps):

  optimizer.zero_grad(set_to_none=True)
  loss_accum = 0.0
  for micro_step in range(grad_accum_steps):
    x,y = train_loader.next_batch()
    x,y = x.to(device),y.to(device)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
      _,loss = model(x,y)
    loss = loss / grad_accum_steps
    loss_accum += loss.detach() ## as the loss is the result of all the compounded functions, with detach we make the scalar a leaf node
    loss.backward()

  norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
  lr = get_lr(step)
  for param_group in optimizer.param_groups:
    param_group['lr'] = lr
  optimizer.step()
  print(f"Step {i}, loss: {loss.item()}, norm:{norm:.4f}}")

Step 0, loss: 10.92757797241211
Step 1, loss: 9.715483665466309
Step 2, loss: 8.628612518310547
Step 3, loss: 9.049093246459961
Step 4, loss: 8.456218719482422
Step 5, loss: 8.077333450317383
Step 6, loss: 8.865742683410645
Step 7, loss: 8.731792449951172
Step 8, loss: 7.957657814025879
Step 9, loss: 7.8860392570495605
Step 10, loss: 8.243453979492188
Step 11, loss: 7.0595502853393555
Step 12, loss: 7.667281627655029
Step 13, loss: 7.318122863769531
Step 14, loss: 7.447032928466797
Step 15, loss: 7.180792331695557
Step 16, loss: 7.238762855529785
Step 17, loss: 8.183853149414062
Step 18, loss: 7.125064373016357
Step 19, loss: 7.675030708312988
Step 20, loss: 7.530610084533691
Step 21, loss: 7.617598533630371
Step 22, loss: 6.166983604431152
Step 23, loss: 6.611393928527832
Step 24, loss: 6.556295871734619
Step 25, loss: 6.217983245849609
Step 26, loss: 6.372446060180664
Step 27, loss: 7.405853271484375
Step 28, loss: 6.9015960693359375
Step 29, loss: 6.572569370269775
Step 30, loss: 6.

In [ ]:
print(loss)

tensor(6.5754, device='cuda:0', grad_fn=<NllLossBackward0>)
